# Lezione 23 — Episodic, semantic, preference: non sono intercambiabili

**Come si usa questo notebook.** Come sempre. Prerequisito: Lezione 22
(schema esplicito). Lo schema tratta `type` come una stringa tra quattro
possibili: valido, ma non basta. Da qui in poi (recency, importanza,
contraddizioni) ogni lezione tratta i quattro type in modo **diverso** —
oggi definiamo esattamente come e perche'.

**Cosa saprai fare alla fine:** caratterizzare episodic/semantic/
preference/unknown con criteri concreti, e costruire una tabella di
parametri per type (decadimento, peso di importanza base) che le Lezioni
24-25 useranno direttamente.

## Parte 1 — Teoria: stesso schema, comportamento diverso

I quattro `type` del progetto non sono etichette intercambiabili di uno
stesso fenomeno: descrivono **relazioni diverse col tempo e con
l'aggiornamento**.

- **Episodic** — un evento specifico, ancorato a un momento e spesso a un
  luogo ("Marco ha visitato Glasgow il 3 luglio"). E' vero per sempre come
  fatto storico, ma la sua **rilevanza** per capire l'utente *oggi* cala
  con il tempo: un viaggio di due anni fa conta meno di uno della settimana
  scorsa per decidere cosa proporre ora. Decade **velocemente**.
- **Semantic** — un fatto generale, spesso senza un momento specifico
  ("la policy Wi-Fi dell'ufficio e' cambiata", "l'utente lavora nel
  team vendite"). Resta rilevante finche' resta vero: non decade con il
  tempo passato dalla sua registrazione, ma puo' diventare **obsoleto** se
  il fatto stesso cambia (tema della Lezione 29, contraddizioni). Decade
  **lentamente**, o non decade affatto per il solo passare del tempo.
- **Preference** — un'opinione, un'abitudine, un'impostazione ("l'utente
  preferisce riunioni brevi"). Non ha una scadenza temporale intrinseca
  (una preferenza espressa un anno fa puo' valere ancora oggi), ma puo'
  essere **sostituita** da un'espressione piu' recente della stessa
  preferenza (tema della Lezione 29). Decade **lentamente**, va pero'
  sempre controllata per sovrascritture.
- **Unknown** — il classificatore (Lezioni 15-17) non ha assegnato un type
  con sicurezza, o il dato mancava (Lezione 1: imputato come costante).
  Senza sapere *cosa* rappresenta, non si puo' assumere ne' un decadimento
  veloce ne' uno lento: si tratta con un peso di importanza **ridotto** di
  default, finche' non viene riclassificata o corretta a mano.

Questa caratterizzazione si traduce in due numeri per type, che le
prossime due lezioni useranno direttamente: un **half-life** (giorni dopo
i quali il contributo di recency si dimezza — Lezione 24) e un **peso di
importanza base** (quanto il type stesso, a prescindere dal contenuto,
alza o abbassa l'importanza — Lezione 25).

In [1]:
PARAMETRI_TIPO = {
    #                  half_life_giorni  peso_importanza_base
    'episodic':   {'half_life_giorni': 30,  'peso_importanza_base': 1.0},
    'semantic':   {'half_life_giorni': 365, 'peso_importanza_base': 1.1},
    'preference': {'half_life_giorni': 180, 'peso_importanza_base': 1.2},
    'unknown':    {'half_life_giorni': 60,  'peso_importanza_base': 0.7},
}

for tipo, parametri in PARAMETRI_TIPO.items():
    print(f"{tipo:12s} half-life={parametri['half_life_giorni']:>3d}gg  "
          f"peso_base={parametri['peso_importanza_base']}")

episodic     half-life= 30gg  peso_base=1.0
semantic     half-life=365gg  peso_base=1.1
preference   half-life=180gg  peso_base=1.2
unknown      half-life= 60gg  peso_base=0.7


Leggi la tabella con occhio critico, non come una verita' assoluta: sono
**scelte di design dichiarate**, non numeri misurati da uno studio — il
punto pedagogico e' che vanno scelte in base al comportamento atteso del
type (episodic con half-life corto perche' decade in fretta, semantic
lungo perche' un fatto resta vero, unknown con peso ridotto perche' non
sappiamo di cosa si tratta), non lasciate uguali per pigrizia. In un
sistema reale andrebbero calibrate su dati d'uso veri (quali memorie gli
utenti trovano ancora utili dopo N giorni), fuori scope qui.

## Parte 2 — Esercizio guidato

Il tuo compito: dato il testo `"The office parking policy changed."`,
decidi a occhio (senza eseguire il classificatore) se e' piu' vicino a
episodic, semantic o preference, poi verifica cosa risponde il criterio
della Parte 1 (ancorato a un evento specifico? e' un fatto generale? e'
un'opinione personale?).

In [2]:
# Scrivi qui: assegna a mano una variabile `tipo_scelto` con la tua risposta
# ('episodic', 'semantic' o 'preference') e stampa il ragionamento.

pass

### Soluzione spiegata

`"The office parking policy changed."` non e' ancorato a un momento
specifico dell'utente (non e' "cosa e' successo", come una visita o una
prenotazione), e non e' un'opinione personale — e' un **fatto generale**
sull'organizzazione, vero finche' non cambia di nuovo: **semantic**. Il
criterio operativo (Parte 1) e' proprio questo: chiediti prima "e' un
evento con un momento preciso?", poi "e' un'opinione di qualcuno?" — se
entrambe le risposte sono no, di solito e' semantic.

In [3]:
tipo_scelto = 'semantic'
ancorato_a_evento = False       # non descrive "cosa e' successo" a un momento preciso
e_unopinione = False            # non esprime una preferenza personale
print(f'tipo scelto: {tipo_scelto}')
assert tipo_scelto == 'semantic'

tipo scelto: semantic


## Parte 3 — Il progetto: Memory AI Lab, passo 23 — parametri per tipo, applicati

Applichiamo `PARAMETRI_TIPO` a tutto il train set: ogni memoria riceve i
suoi due parametri in base al `type` gia' assegnato (Lezioni 1, 15-17). Il
risultato e' un'unica tabella pronta per la Lezione 24 (recency) e la
Lezione 25 (importanza) — non ricalcoleranno questi numeri, li
useranno.

In [4]:
import pandas as pd
from pathlib import Path

processed = Path('..') / 'datasets' / 'processed'
train = pd.read_csv(processed / 'memory_train.csv')

train['half_life_giorni'] = train['type'].map(lambda t: PARAMETRI_TIPO[t]['half_life_giorni'])
train['peso_importanza_base'] = train['type'].map(lambda t: PARAMETRI_TIPO[t]['peso_importanza_base'])

riepilogo = train.groupby('type')[['half_life_giorni', 'peso_importanza_base']].first()
conteggi = train['type'].value_counts()
riepilogo['memorie'] = conteggi
print(riepilogo)

assert train['half_life_giorni'].notna().all()  # ogni memoria ha un type noto (Lezione 22)
assert train['peso_importanza_base'].notna().all()

            half_life_giorni  peso_importanza_base  memorie
type                                                       
episodic                  30                   1.0      108
preference               180                   1.2       61
semantic                 365                   1.1       43
unknown                   60                   0.7        1


## Cosa hai imparato — e le prossime due lezioni partono da qui

- I quattro `type` non sono etichette equivalenti: **episodic** decade in
  fretta (ancorato a un evento), **semantic** decade lentamente (un fatto
  generale), **preference** decade lentamente ma va controllata per
  sovrascritture, **unknown** riceve un peso ridotto per mancanza di
  informazione.
- Tradurre questa caratterizzazione in **parametri numerici dichiarati**
  (half-life, peso base) per type rende il comportamento differenziato
  eseguibile, non solo descritto a parole.
- I parametri di questa lezione sono scelte di design esplicite, non
  misure: in un sistema reale andrebbero calibrati su dati d'uso reali.

## Quiz

1. Perche' `episodic` ha un half-life piu' corto di `semantic` in
   `PARAMETRI_TIPO`?
2. Una preferenza espressa un anno fa e' ancora valida di default: perche'
   allora il testo dice che va comunque "controllata per sovrascritture"?
3. Perche' `unknown` riceve un peso di importanza base ridotto invece che
   pari alla media degli altri type?

<details>
<summary><b>Apri le risposte</b></summary>

1. Perche' un episodic descrive un evento specifico: la sua rilevanza per
   capire l'utente *oggi* cala man mano che l'evento si allontana nel
   tempo. Un semantic descrive un fatto generale che, salvo cambiamenti,
   resta vero indipendentemente da quanto tempo e' passato dalla sua
   registrazione.
2. Perche' una preferenza puo' cambiare senza che la vecchia affermazione
   smetta di "sembrare" valida: se l'utente esprime una preferenza nuova
   che contraddice la precedente, la vecchia va aggiornata o sostituita
   (Lezione 29) — non basta che il tempo passi, serve un controllo
   esplicito di coerenza.
3. Perche' senza sapere cosa rappresenta la memoria (il classificatore non
   ha assegnato un type con sicurezza, o il dato mancava), non c'e' base
   per assumere ne' un decadimento lento ne' un peso alto: un peso ridotto
   riflette onestamente l'incertezza, invece di trattare "non so" come
   equivalente a "so, ed e' nella media".
</details>

## Fonti

- pandas, `Series.map`:
  https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html
- pandas, `DataFrame.groupby`:
  https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html

La caratterizzazione di episodic/semantic/preference/unknown e i valori
di `PARAMETRI_TIPO` sono scelte di design per il Memory AI Lab, non da una
fonte esterna — dichiarato esplicitamente nella Parte 1, non presentato
come uno standard misurato.

## Prossima lezione

I due numeri di oggi (half-life, peso base) sono ingredienti dichiarati,
non ancora usati: la prossima lezione costruisce la formula di
**decadimento temporale** che consuma l'half-life, con curve concrete per
ogni type.